# Sentiment-Augmented DRL Trading — FINAL v3
## All bugs fixed + Anti-overfitting + Realistic results
```
Key fixes in v3:
  - NaN bug fixed: removed LayerNorm, added orthogonal init + NaN guards
  - HF token passed explicitly to all from_pretrained() calls
  - Val split: 2019-2022 (includes 2022 bear market -65%)
  - HP tuning: 2 configs per agent
  - Checkpoint rejection if val_CR > 500%
```

In [ ]:
!pip install -q -U transformers peft datasets yfinance gymnasium scikit-learn torchao
import torch, transformers
print(f"Torch {torch.__version__} | CUDA {torch.cuda.is_available()}")
print("Restart kernel then Run All")

In [ ]:
ASSET      = "TSLA"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
TRAIN_END  = "2018-12-31"
VAL_START  = "2019-01-01"
VAL_END    = "2022-12-31"
TEST_START = "2023-01-01"
MAX_EP_SEARCH    = 150
MAX_EP_FINAL     = 400
EVAL_EVERY       = 25
PATIENCE         = 6
MAX_STEPS_PER_EP = 252
MAX_REALISTIC_VAL_CR = 500.0
REPO = "vijay5799760/Qwen2.5-3B-SEntFiN-LoRA-HPtuned"
BASE = "Qwen/Qwen2.5-3B-Instruct"
print(f"Asset:{ASSET} Device:{DEVICE}")
print(f"Train:up to {TRAIN_END} | Val:{VAL_START}-{VAL_END} | Test:{TEST_START}+")

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("HF login OK")

In [ ]:
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import gymnasium as gym
import yfinance as yf
import json, os, gc, copy, hashlib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42)

In [ ]:
def load_prices(ticker):
    df = yf.download(ticker, start="2010-01-01", auto_adjust=True, progress=False)
    df = df[["Close","Volume"]].copy()
    df.columns = ["close","volume"]
    df.index = pd.to_datetime(df.index).normalize()
    return df.dropna().reset_index().rename(columns={"Date":"date"})

prices = load_prices(ASSET)
prices["date"] = pd.to_datetime(prices["date"])
train_df = prices[prices["date"] <= TRAIN_END].copy()
val_df   = prices[(prices["date"] >= VAL_START) & (prices["date"] <= VAL_END)].copy()
test_df  = prices[prices["date"] >= TEST_START].copy()

for name, df in [("Train",train_df),("Val",val_df),("Test",test_df)]:
    if len(df) < 2: print(f"{name}: no data"); continue
    bnh = (df["close"].iloc[-1]/df["close"].iloc[0]-1)*100
    print(f"{name}: {len(df)} days | B&H={bnh:.1f}% ({df['date'].min().date()} to {df['date'].max().date()})")

print("\nVal 2019-2022 regime:")
print("  2019:+26%  2020:+743% (COVID crash+recovery)  2021:+50%  2022:-65% (bear)")
print("  Agent that just stays LONG loses -65% in 2022 on val -> forces robust policy")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print(f"Loading SEntFiN on {DEVICE}...")

# FIX: pass token= explicitly to all from_pretrained calls (avoids 401 errors)
sent_tok   = AutoTokenizer.from_pretrained(REPO, token=HF_TOKEN)
sent_base  = AutoModelForCausalLM.from_pretrained(
    BASE, dtype=torch.float16, device_map="auto", token=HF_TOKEN
)
sent_model = PeftModel.from_pretrained(sent_base, REPO, token=HF_TOKEN)
sent_model.eval()

ASSET_NAME = {"TSLA":"Tesla (TSLA)","BTC-USD":"Bitcoin (BTC)"}.get(ASSET,ASSET)
SYSTEM = (f"You are a financial sentiment analyst. Classify the sentiment towards {ASSET_NAME}. "
          f"Reply with exactly one word: positive, negative, or neutral.")

lids={}
for lbl in ["positive","negative","neutral"]:
    ids=[]
    for v in [lbl," "+lbl,lbl.capitalize()," "+lbl.capitalize()]:
        ids.extend(sent_tok.encode(v,add_special_tokens=False))
    lids[lbl]=list(set(ids))
n_pos=len(lids["positive"]); n_neg=len(lids["negative"])
all_ids=lids["positive"]+lids["negative"]+lids["neutral"]

def score_article(text):
    msgs=[{"role":"system","content":SYSTEM},
          {"role":"user","content":f"Headline: {' '.join(text.split()[:300])}\nCompany: {ASSET_NAME}\nSentiment:"}]
    try: prompt=sent_tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    except: prompt=f"{SYSTEM}\n\nHeadline: {text[:300]}\nCompany: {ASSET_NAME}\nSentiment:"
    inp=sent_tok(prompt,return_tensors="pt",truncation=True,max_length=400).to(DEVICE)
    with torch.no_grad(): logits=sent_model(**inp).logits[0,-1,:]
    probs=torch.softmax(logits[all_ids],dim=0).cpu()
    return float(probs[:n_pos].sum()-probs[n_pos:n_pos+n_neg].sum())

print("Scorer ready")
print(f"Test: 'Infosys opens AI lab' -> {score_article('Infosys opens AI lab with Nvidia'):.3f}")

In [ ]:
import hashlib
from datasets import load_dataset

CACHE_FILE = Path(f"/kaggle/working/sent_cache_{ASSET}.json")
SENT_CSV   = Path(f"/kaggle/working/daily_sentiment_{ASSET}.csv")
MAX_PER_SOURCE = 5000
KEYWORDS = {"TSLA":["tesla","tsla","elon","musk"],"BTC-USD":["bitcoin","btc","crypto"]}.get(ASSET,[ASSET.lower()])

def hash_text(t): return hashlib.md5(t[:500].encode()).hexdigest()
def safe_ts(ts):
    try:
        t=pd.Timestamp(ts) if not isinstance(ts,pd.Timestamp) else ts
        if t.tzinfo is not None: t=t.tz_localize(None)
        return t.normalize()
    except: return None
def is_relevant(text): return any(kw in (text or "").lower() for kw in KEYWORDS)

if SENT_CSV.exists():
    print(f"Loading pre-scored CSV (instant)...")
    saved=pd.read_csv(SENT_CSV,index_col=0,parse_dates=True)
    saved.index=pd.to_datetime(saved.index).tz_localize(None)
    daily_sent=saved["score"]
    print(f"Loaded {len(daily_sent)} days")
else:
    print("CSV not found - scoring articles...")
    cache={}
    if CACHE_FILE.exists():
        with open(CACHE_FILE) as f: cache=json.load(f)
    all_articles=[]
    for ds_name,use_stream,max_n in [
        ("Brianferrell787/financial-news-multisource",True,MAX_PER_SOURCE),
        ("ashraq/financial-news-articles",False,MAX_PER_SOURCE),
        ("Johnson8187/us_economic_news_article",False,MAX_PER_SOURCE),
    ]:
        try:
            ds=load_dataset(ds_name,split="train",streaming=use_stream)
            n=0
            for ex in ds:
                t=ex.get("text") or ex.get("title") or ex.get("content") or ""
                if not is_relevant(t): continue
                d=safe_ts(ex.get("date") or ex.get("publishedAt") or "")
                if d is None: d=pd.Timestamp("2016-01-01")+pd.Timedelta(days=n*2)
                all_articles.append({"text":str(t)[:400],"date":d}); n+=1
                if n>=max_n: break
            print(f"  {ds_name}: {n} articles")
        except Exception as e: print(f"  {ds_name}: {e}")

    daily_scores={}
    for i,row in enumerate(all_articles):
        key=hash_text(row["text"])
        score=cache.get(key)
        if score is None: score=score_article(row["text"]); cache[key]=score
        daily_scores.setdefault(row["date"],[]).append(score)
        if (i+1)%200==0:
            with open(CACHE_FILE,"w") as f: json.dump(cache,f)
            print(f"  {i+1}/{len(all_articles)}")
    with open(CACHE_FILE,"w") as f: json.dump(cache,f)
    daily_mean={d:sum(v)/len(v) for d,v in daily_scores.items()}
    daily_sent=pd.Series(daily_mean).sort_index()
    out_df=daily_sent.reset_index(); out_df.columns=["date","score"]
    out_df.to_csv(SENT_CSV,index=False)
    print(f"Saved {SENT_CSV}")

def align(df): return pd.Series(df["date"].map(daily_sent).fillna(0.0).tolist(),index=df["date"].tolist())
train_sent=align(train_df); val_sent=align(val_df); test_sent=align(test_df)
for n,s in [("Train",train_sent),("Val",val_sent),("Test",test_sent)]:
    print(f"  {n} coverage:{(s!=0).mean():.1%} mean={s.mean():+.3f}")
del sent_model,sent_base; gc.collect(); torch.cuda.empty_cache()
print("GPU cleared")

In [ ]:
def build_states(df, sentiment, warmup=60):
    df=df.copy().sort_values("date").reset_index(drop=True)
    c,v=df["close"],df["volume"]
    ema10=c.ewm(span=10,adjust=False).mean(); ema50=c.ewm(span=50,adjust=False).mean()
    delta=c.diff(); gain=delta.clip(lower=0).ewm(span=14,adjust=False).mean()
    loss=(-delta.clip(upper=0)).ewm(span=14,adjust=False).mean()
    rsi14=100-(100/(1+gain/(loss+1e-10)))
    ema12=c.ewm(span=12,adjust=False).mean(); ema26=c.ewm(span=26,adjust=False).mean()
    macd=ema12-ema26; sig=macd.ewm(span=9,adjust=False).mean()
    sma20=c.rolling(20).mean(); std20=c.rolling(20).std()
    df["log_ret"]=np.log(c/c.shift(1)+1e-10); df["daily_ret"]=c.pct_change()
    df["ema10_f"]=ema10/c-1; df["ema50_f"]=ema50/c-1
    df["rsi_f"]=rsi14/50-1; df["macd_f"]=(macd-sig)/(c+1e-10)
    df["boll_f"]=(c-sma20)/(2*std20+1e-10); df["vol_f"]=np.log(v/v.shift(1)+1e-10)
    dates=pd.DatetimeIndex(df["date"])
    df["dow_sin"]=np.sin(2*np.pi*dates.dayofweek/7); df["dow_cos"]=np.cos(2*np.pi*dates.dayofweek/7)
    df["dom_sin"]=np.sin(2*np.pi*dates.day/31); df["dom_cos"]=np.cos(2*np.pi*dates.day/31)
    df["moy_sin"]=np.sin(2*np.pi*dates.month/12); df["moy_cos"]=np.cos(2*np.pi*dates.month/12)
    raw_sent=df["date"].map(dict(zip(sentiment.index,sentiment.values))).fillna(0.0)
    df["sentiment"]=raw_sent.ewm(span=7,adjust=False).mean()
    df=df.iloc[warmup:].dropna().reset_index(drop=True)
    for col in ["log_ret","ema10_f","ema50_f","rsi_f","macd_f","boll_f","vol_f"]:
        mu=df[col].rolling(252,min_periods=20).mean()
        sd=df[col].rolling(252,min_periods=20).std().clip(lower=1e-8)
        df[col]=(df[col]-mu)/sd
    S=np.zeros((len(df),15),np.float32)
    S[:,0]=df["log_ret"]; S[:,1]=0.0
    S[:,2]=df["dow_sin"]; S[:,3]=df["dow_cos"]
    S[:,4]=df["dom_sin"]; S[:,5]=df["dom_cos"]
    S[:,6]=df["moy_sin"]; S[:,7]=df["moy_cos"]
    S[:,8]=df["ema10_f"]; S[:,9]=df["ema50_f"]
    S[:,10]=df["rsi_f"]; S[:,11]=df["macd_f"]
    S[:,12]=df["boll_f"]; S[:,13]=df["vol_f"]
    S[:,14]=df["sentiment"]
    S=np.nan_to_num(np.clip(S,-5,5),nan=0.0)
    return S, df["daily_ret"].values.astype(np.float32)

WARMUP=60
val_ext =pd.concat([train_df.tail(WARMUP),val_df]).reset_index(drop=True)
test_ext=pd.concat([val_df.tail(WARMUP),test_df]).reset_index(drop=True)
all_sent=pd.concat([train_sent,val_sent,test_sent])
train_S,train_R=build_states(train_df,train_sent)
val_S,val_R    =build_states(val_ext,all_sent)
test_S,test_R  =build_states(test_ext,all_sent)
print(f"States: Train={len(train_S)} Val={len(val_S)} Test={len(test_S)}")

In [ ]:
class TradingEnv(gym.Env):
    P={0:-1,1:0,2:1}
    def __init__(self,S,R,rand=True,cost=True):
        super().__init__()
        self.S=S.astype(np.float32); self.R=R.astype(np.float32)
        self.T=len(S); self.rand=rand; self.tc=0.002 if cost else 0.0
        self.action_space=gym.spaces.Discrete(3)
        self.observation_space=gym.spaces.Box(-np.inf,np.inf,(15,),np.float32)
        self.reset()
    def reset(self,seed=None,options=None):
        super().reset(seed=seed)
        self.t=int(np.random.randint(1,max(2,self.T-100))) if self.rand else 1
        self.val=100_000.0; self.prev=0; self.vals=[self.val]; self.acts=[0]
        return self._obs(),{}
    def _obs(self):
        s=self.S[self.t].copy(); s[1]=float(self.prev)
        return np.nan_to_num(s,nan=0.0)
    def step(self,action):
        pos=self.P[action]; ret=float(self.R[self.t])
        if not np.isfinite(ret): ret=0.0
        cost=self.tc if pos!=self.prev else 0.0
        nv=max(self.val*(1+pos*ret)*(1-cost),100.0)
        raw=np.log(nv/self.val+1e-10)-np.log(1+ret+1e-10)
        alpha_r=float(np.clip(raw,-0.03,0.03))
        self.val=nv; self.prev=pos; self.t+=1
        self.vals.append(nv); self.acts.append(pos)
        done=(self.t>=self.T-1)
        return (self._obs() if not done else np.zeros(15,np.float32)),alpha_r,done,False,{}
    def metrics(self):
        v=np.array(self.vals); r=np.diff(v)/v[:-1]
        sr=float(np.mean(r)/(np.std(r)+1e-10)*np.sqrt(252))
        cr=float((v[-1]/v[0]-1)*100)
        pk=np.maximum.accumulate(v); mdd=float(abs(((v-pk)/pk).min())*100)
        return {"CR%":round(cr,2),"SR":round(sr,3),"MDD%":round(mdd,2)}

def make_env(S,R,rand=True,cost=True): return TradingEnv(S,R,rand,cost)

In [ ]:
# FIX: removed LayerNorm (caused NaN when all ReLU=0 after Dropout)
# FIX: orthogonal init with small gain (stable from step 1)

def make_ffnn(inp,hid,out,nl=2,drop=0.3):
    lays=[nn.Linear(inp,hid),nn.ReLU(),nn.Dropout(drop)]
    for _ in range(nl-1): lays+=[nn.Linear(hid,hid),nn.ReLU(),nn.Dropout(drop)]
    lays.append(nn.Linear(hid,out))
    net=nn.Sequential(*lays)
    for m in net.modules():
        if isinstance(m,nn.Linear):
            nn.init.orthogonal_(m.weight,gain=0.01)
            nn.init.zeros_(m.bias)
    return net

class QNet(nn.Module):
    def __init__(self,hid=128,nl=2): super().__init__(); self.net=make_ffnn(15,hid,3,nl,0.2)
    def forward(self,x): return self.net(x)

class DDPGActor(nn.Module):
    def __init__(self,hid=128,nl=2):
        super().__init__()
        lays=[nn.Linear(15,hid),nn.ReLU(),nn.Dropout(0.2)]
        for _ in range(nl-1): lays+=[nn.Linear(hid,hid),nn.ReLU(),nn.Dropout(0.2)]
        lays+=[nn.Linear(hid,1),nn.Tanh()]
        self.net=nn.Sequential(*lays)
        for m in self.net.modules():
            if isinstance(m,nn.Linear): nn.init.orthogonal_(m.weight,gain=0.01); nn.init.zeros_(m.bias)
    def forward(self,x): return self.net(x)
    def discrete(self,x):
        with torch.no_grad(): v=self.net(x.unsqueeze(0)).item()
        if not np.isfinite(v): v=0.0
        return (0 if v<-1/3 else (2 if v>1/3 else 1)),v

class DDPGCritic(nn.Module):
    def __init__(self,hid=128,nl=2): super().__init__(); self.net=make_ffnn(16,hid,1,nl,0.2)
    def forward(self,s,a): return self.net(torch.cat([s,a],dim=-1))

class ReplayBuffer:
    def __init__(self,cap=10000):
        self.cap=cap; self.ptr=0; self.sz=0
        self.S=np.zeros((cap,15),np.float32); self.A=np.zeros(cap,np.int64)
        self.R=np.zeros(cap,np.float32); self.NS=np.zeros((cap,15),np.float32)
        self.D=np.zeros(cap,np.float32); self.CA=np.zeros(cap,np.float32)
    def push(self,s,a,r,ns,d,ca=0.0):
        i=self.ptr
        self.S[i]=s; self.A[i]=a; self.R[i]=r; self.NS[i]=ns; self.D[i]=d; self.CA[i]=ca
        self.ptr=(i+1)%self.cap; self.sz=min(self.sz+1,self.cap)
    def sample(self,bs,dev):
        i=np.random.choice(self.sz,bs,replace=False)
        return (torch.FloatTensor(self.S[i]).to(dev),torch.LongTensor(self.A[i]).to(dev),
                torch.FloatTensor(self.R[i]).to(dev),torch.FloatTensor(self.NS[i]).to(dev),
                torch.FloatTensor(self.D[i]).to(dev),torch.FloatTensor(self.CA[i]).to(dev))
    def __len__(self): return self.sz

class OUNoise:
    def __init__(self,th=0.15,sig=0.2): self.th=th; self.sig=sig; self.reset()
    def reset(self): self.s=0.0
    def __call__(self): self.s+=self.th*(0-self.s)+self.sig*np.random.randn(); return float(np.clip(self.s,-1,1))

In [ ]:
def _safe_obs(s):
    return np.nan_to_num(np.array(s,dtype=np.float32),nan=0.0,posinf=5.0,neginf=-5.0)

def add_noise(obs,std=0.02):
    obs=_safe_obs(obs)
    return obs+np.random.normal(0,std,obs.shape).astype(np.float32)

class PGAgent:
    def __init__(self,hid=64,nl=1,lr=1e-3,gamma=0.99,ent=0.05,wd=1e-3):
        self.gamma=gamma; self.ent=ent
        self.policy=make_ffnn(15,hid,3,nl,0.3).to(DEVICE)
        self.opt=torch.optim.Adam(self.policy.parameters(),lr=lr,weight_decay=wd)
        self.lp=[]; self.rw=[]; self.en=[]
    def act(self,s):
        s=_safe_obs(s)
        lg=self.policy(torch.FloatTensor(s).to(DEVICE))
        if torch.isnan(lg).any() or torch.isinf(lg).any():
            return np.random.randint(3)
        d=torch.distributions.Categorical(logits=lg); a=d.sample()
        self.lp.append(d.log_prob(a)); self.en.append(d.entropy()); return a.item()
    def store(self,r): self.rw.append(float(r) if np.isfinite(r) else 0.0)
    def update(self):
        if not self.rw: return {}
        G=0.0; ret=[]
        for r in reversed(self.rw): G=r+self.gamma*G; ret.insert(0,G)
        ret=torch.FloatTensor(ret).to(DEVICE)
        if torch.isnan(ret).any(): self.lp=[]; self.rw=[]; self.en=[]; return {}
        ret=(ret-ret.mean())/(ret.std()+1e-8)
        loss=-(torch.stack(self.lp)*ret).mean()-self.ent*torch.stack(self.en).mean()
        if torch.isnan(loss): self.lp=[]; self.rw=[]; self.en=[]; return {}
        self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.policy.parameters(),0.5)
        self.opt.step(); self.lp=[]; self.rw=[]; self.en=[]; return {}
    def greedy(self,s):
        s=_safe_obs(s); self.policy.eval()
        with torch.no_grad(): lg=self.policy(torch.FloatTensor(s).to(DEVICE))
        self.policy.train()
        if torch.isnan(lg).any(): return 1
        return lg.argmax().item()

class PPOAgent:
    def __init__(self,hid=64,nl=1,lr=3e-4,gamma=0.99,lam=0.95,eps=0.2,vc=0.5,ec=0.05,epochs=4,wd=1e-3):
        self.gamma=gamma; self.lam=lam; self.eps=eps; self.vc=vc; self.ec=ec; self.epochs=epochs
        self.backbone=nn.Sequential(nn.Linear(15,hid),nn.ReLU(),nn.Dropout(0.3)).to(DEVICE)
        self.actor=nn.Linear(hid,3).to(DEVICE); self.critic=nn.Linear(hid,1).to(DEVICE)
        for m in list(self.backbone.modules())+[self.actor,self.critic]:
            if isinstance(m,nn.Linear): nn.init.orthogonal_(m.weight,gain=0.01); nn.init.zeros_(m.bias)
        params=list(self.backbone.parameters())+list(self.actor.parameters())+list(self.critic.parameters())
        self.opt=torch.optim.Adam(params,lr=lr,weight_decay=wd)
        self.ss=[]; self.aa=[]; self.lp=[]; self.rr=[]; self.vv=[]; self.dd=[]
    def _fwd(self,x):
        h=self.backbone(x); return self.actor(h),self.critic(h).squeeze(-1)
    def act(self,s):
        s=_safe_obs(s); st=torch.FloatTensor(s).to(DEVICE)
        with torch.no_grad(): lg,val=self._fwd(st)
        if torch.isnan(lg).any():
            self.ss.append(s); self.aa.append(1); self.lp.append(0.0); self.vv.append(0.0); return 1
        d=torch.distributions.Categorical(logits=lg); a=d.sample()
        self.ss.append(s); self.aa.append(a.item())
        self.lp.append(d.log_prob(a).item()); self.vv.append(float(val)); return a.item()
    def store(self,r,done): self.rr.append(float(r) if np.isfinite(r) else 0.0); self.dd.append(done)
    def update(self,last_val=0.0):
        if len(self.rr)<2:
            self.ss=[]; self.aa=[]; self.lp=[]; self.rr=[]; self.vv=[]; self.dd=[]; return {}
        vals=self.vv+[last_val]; adv=[]; g=0.0
        for t in reversed(range(len(self.rr))):
            mask=1-self.dd[t]; delta=self.rr[t]+self.gamma*vals[t+1]*mask-vals[t]
            g=delta+self.gamma*self.lam*mask*g; adv.insert(0,g)
        adv=torch.FloatTensor(adv).to(DEVICE); adv=(adv-adv.mean())/(adv.std()+1e-8)
        ret=adv+torch.FloatTensor(self.vv).to(DEVICE)
        st=torch.FloatTensor(np.array(self.ss)).to(DEVICE)
        at=torch.LongTensor(self.aa).to(DEVICE); olp=torch.FloatTensor(self.lp).to(DEVICE)
        all_p=list(self.backbone.parameters())+list(self.actor.parameters())+list(self.critic.parameters())
        for _ in range(self.epochs):
            lg,nv=self._fwd(st)
            if torch.isnan(lg).any(): break
            d=torch.distributions.Categorical(logits=lg); ratio=(d.log_prob(at)-olp).exp()
            al=-torch.min(ratio*adv,torch.clamp(ratio,1-self.eps,1+self.eps)*adv).mean()
            loss=al+self.vc*F.mse_loss(nv,ret)-self.ec*d.entropy().mean()
            if torch.isnan(loss): break
            self.opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(all_p,0.5); self.opt.step()
        self.ss=[]; self.aa=[]; self.lp=[]; self.rr=[]; self.vv=[]; self.dd=[]; return {}
    def greedy(self,s):
        s=_safe_obs(s); self.backbone.eval(); self.actor.eval()
        with torch.no_grad(): lg,_=self._fwd(torch.FloatTensor(s).to(DEVICE))
        self.backbone.train(); self.actor.train()
        if torch.isnan(lg).any(): return 1
        return lg.argmax().item()

class DQLAgent:
    def __init__(self,hid=128,nl=2,lr=1e-3,gamma=0.99,buf=10000,bs=64,tgt=100,
                 eps_s=1.0,eps_e=0.05,eps_d=5000,wd=1e-3):
        self.gamma=gamma; self.bs=bs; self.tgt=tgt; self.steps=0
        self.eps=eps_s; self.eps_s=eps_s; self.eps_e=eps_e; self.eps_d=eps_d
        self.online=QNet(hid,nl).to(DEVICE); self.target=QNet(hid,nl).to(DEVICE)
        self.target.load_state_dict(self.online.state_dict()); self.target.eval()
        self.opt=torch.optim.Adam(self.online.parameters(),lr=lr,weight_decay=wd)
        self.buf=ReplayBuffer(buf)
    def act(self,s):
        s=_safe_obs(s)
        self.eps=max(self.eps_e,self.eps_s-self.steps/self.eps_d*(self.eps_s-self.eps_e))
        if np.random.rand()<self.eps: return np.random.randint(3)
        self.online.eval()
        with torch.no_grad(): q=self.online(torch.FloatTensor(s).unsqueeze(0).to(DEVICE))
        self.online.train()
        if torch.isnan(q).any(): return np.random.randint(3)
        return q.argmax().item()
    def store(self,s,a,r,ns,d):
        self.buf.push(_safe_obs(s),a,float(r) if np.isfinite(r) else 0.0,_safe_obs(ns),d)
        self.steps+=1
    def update(self):
        if len(self.buf)<self.bs: return {}
        s,a,r,ns,d,_=self.buf.sample(self.bs,DEVICE)
        with torch.no_grad():
            na=self.online(ns).argmax(1,keepdim=True); tq=self.target(ns).gather(1,na).squeeze()
            tgt=r+self.gamma*tq*(1-d)
        cq=self.online(s).gather(1,a.unsqueeze(1)).squeeze(); loss=F.smooth_l1_loss(cq,tgt)
        if torch.isnan(loss): return {}
        self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.online.parameters(),0.5); self.opt.step()
        if self.steps%self.tgt==0: self.target.load_state_dict(self.online.state_dict())
        return {}
    def greedy(self,s):
        s=_safe_obs(s); self.online.eval()
        with torch.no_grad(): q=self.online(torch.FloatTensor(s).unsqueeze(0).to(DEVICE))
        self.online.train()
        if torch.isnan(q).any(): return 1
        return q.argmax().item()

class DDPGAgent:
    def __init__(self,hid=128,nl=2,la=1e-4,lc=1e-3,gamma=0.99,tau=0.005,
                 buf=10000,bs=64,ou_t=0.15,ou_s=0.2,wd=1e-3):
        self.gamma=gamma; self.tau=tau; self.bs=bs
        self.actor=DDPGActor(hid,nl).to(DEVICE); self.critic=DDPGCritic(hid,nl).to(DEVICE)
        self.actor_t=DDPGActor(hid,nl).to(DEVICE); self.critic_t=DDPGCritic(hid,nl).to(DEVICE)
        self.actor_t.load_state_dict(self.actor.state_dict()); self.critic_t.load_state_dict(self.critic.state_dict())
        self.actor_t.eval(); self.critic_t.eval()
        self.opt_a=torch.optim.Adam(self.actor.parameters(),lr=la,weight_decay=wd)
        self.opt_c=torch.optim.Adam(self.critic.parameters(),lr=lc,weight_decay=wd)
        self.buf=ReplayBuffer(buf); self.noise=OUNoise(ou_t,ou_s)
    def _soft(self,o,t):
        for p,pt in zip(o.parameters(),t.parameters()): pt.data.copy_(self.tau*p.data+(1-self.tau)*pt.data)
    def act(self,s,explore=True):
        s=_safe_obs(s); st=torch.FloatTensor(s).to(DEVICE)
        self.actor.eval(); disc,ca=self.actor.discrete(st); self.actor.train()
        if explore: ca=float(np.clip(ca+self.noise(),-1,1)); disc=0 if ca<-1/3 else (2 if ca>1/3 else 1)
        return disc,ca
    def store(self,s,a,r,ns,d,ca):
        self.buf.push(_safe_obs(s),a,float(r) if np.isfinite(r) else 0.0,_safe_obs(ns),d,float(ca))
    def update(self):
        if len(self.buf)<self.bs: return {}
        s,a,r,ns,d,ca=self.buf.sample(self.bs,DEVICE)
        with torch.no_grad():
            na=self.actor_t(ns); tq=self.critic_t(ns,na).squeeze(); tgt=r+self.gamma*tq*(1-d)
        cq=self.critic(s,ca.unsqueeze(1)).squeeze(); cl=F.mse_loss(cq,tgt)
        if torch.isnan(cl): return {}
        self.opt_c.zero_grad(); cl.backward(); nn.utils.clip_grad_norm_(self.critic.parameters(),0.5); self.opt_c.step()
        pa=self.actor(s); al=-self.critic(s,pa).mean()
        if torch.isnan(al): return {}
        self.opt_a.zero_grad(); al.backward(); nn.utils.clip_grad_norm_(self.actor.parameters(),0.5); self.opt_a.step()
        self._soft(self.actor,self.actor_t); self._soft(self.critic,self.critic_t); return {}
    def greedy(self,s): disc,_=self.act(s,explore=False); return disc

In [ ]:
def sharpe(vals):
    r=np.diff(np.array(vals))/np.array(vals[:-1])
    return float(np.mean(r)/(np.std(r)+1e-10)*np.sqrt(252)) if len(r)>1 else 0.0

def run_eval(agent,S,R):
    env=make_env(S,R,rand=False,cost=True)
    obs,_=env.reset(); done=False
    while not done: obs,_,done,_,_=env.step(agent.greedy(obs))
    return sharpe(env.vals),env.metrics()

def create_agent(name,cfg):
    if name=="PG":   return PGAgent(**cfg)
    elif name=="PPO": return PPOAgent(**cfg)
    elif name=="DQL": return DQLAgent(**cfg)
    elif name=="DDPG": return DDPGAgent(**cfg)

def save_state(name,agent):
    if name=="PG":   return copy.deepcopy(agent.policy.state_dict())
    elif name=="PPO": return (copy.deepcopy(agent.backbone.state_dict()),
                              copy.deepcopy(agent.actor.state_dict()),
                              copy.deepcopy(agent.critic.state_dict()))
    elif name=="DQL": return copy.deepcopy(agent.online.state_dict())
    elif name=="DDPG": return (copy.deepcopy(agent.actor.state_dict()),
                               copy.deepcopy(agent.critic.state_dict()))

def load_state(name,agent,st):
    if name=="PG":   agent.policy.load_state_dict(st)
    elif name=="PPO": agent.backbone.load_state_dict(st[0]); agent.actor.load_state_dict(st[1]); agent.critic.load_state_dict(st[2])
    elif name=="DQL": agent.online.load_state_dict(st)
    elif name=="DDPG": agent.actor.load_state_dict(st[0]); agent.critic.load_state_dict(st[1])

def run_episode(agent,name,env):
    obs,_=env.reset(); done=False; tot=0.0; steps=0; lv=0.0
    if name=="PG":
        while not done:
            a=agent.act(add_noise(obs)); obs,r,done,_,_=env.step(a); agent.store(r); tot+=r
        agent.update()
    elif name=="PPO":
        while not done:
            a=agent.act(add_noise(obs)); obs,r,done,_,_=env.step(a); agent.store(r,done); tot+=r
        if not done:
            with torch.no_grad(): _,lv=agent._fwd(torch.FloatTensor(_safe_obs(obs)).to(DEVICE)); lv=float(lv)
        agent.update(lv)
    elif name=="DQL":
        while not done and steps<MAX_STEPS_PER_EP:
            a=agent.act(add_noise(obs)); no,r,done,_,_=env.step(a)
            agent.store(obs,a,r,no,done); agent.update(); obs=no; tot+=r; steps+=1
    elif name=="DDPG":
        agent.noise.reset()
        while not done and steps<MAX_STEPS_PER_EP:
            a,ca=agent.act(add_noise(obs)); no,r,done,_,_=env.step(a)
            agent.store(obs,a,r,no,done,ca); agent.update(); obs=no; tot+=r; steps+=1
    return tot

def train_agent(name,cfg,max_ep,label=""):
    agent=create_agent(name,cfg)
    best_sr=-np.inf; best_st=None; best_cr=0.0; no_imp=0
    for ep in range(1,max_ep+1):
        env=make_env(train_S,train_R,rand=True,cost=True)
        try: tot=run_episode(agent,name,env)
        except Exception as e: print(f"  ep {ep} error: {e}"); continue
        if ep%EVAL_EVERY==0:
            try: sr,m=run_eval(agent,val_S,val_R)
            except: sr=0.0; m={"CR%":0,"SR":0,"MDD%":0}
            val_cr=m["CR%"]
            is_realistic=val_cr < MAX_REALISTIC_VAL_CR
            flag="OK" if is_realistic else "OVERFIT-skip"
            print(f"  [{name}{label}] ep={ep:4d} | val_SR={sr:.3f} | val_CR={val_cr:.1f}% [{flag}]")
            if sr>best_sr and is_realistic:
                best_sr=sr; best_cr=val_cr; best_st=save_state(name,agent); no_imp=0
            else:
                no_imp+=1
                if no_imp>=PATIENCE: print(f"  [{name}{label}] Early stop ep={ep}"); break
    if best_st: load_state(name,agent,best_st)
    print(f"  [{name}{label}] Done: best val_SR={best_sr:.3f} val_CR={best_cr:.1f}%")
    return agent,best_sr,best_cr

In [ ]:
print("="*60)
print("PHASE 1: HP Search (val=2019-2022 includes bear market)")
print("="*60)

HP_GRID={
    "PG":  [
        {"hid":64, "nl":1,"lr":1e-3,"gamma":0.99,"ent":0.05,"wd":1e-3},
        {"hid":128,"nl":2,"lr":5e-4,"gamma":0.99,"ent":0.1, "wd":1e-3},
    ],
    "PPO": [
        {"hid":64, "nl":1,"lr":3e-4,"gamma":0.99,"lam":0.95,"eps":0.2,"vc":0.5,"ec":0.05,"epochs":4,"wd":1e-3},
        {"hid":128,"nl":1,"lr":1e-4,"gamma":0.99,"lam":0.95,"eps":0.1,"vc":0.5,"ec":0.1, "epochs":4,"wd":1e-3},
    ],
    "DQL": [
        {"hid":128,"nl":2,"lr":1e-3,"gamma":0.99,"wd":1e-3},
        {"hid":256,"nl":2,"lr":5e-4,"gamma":0.99,"wd":1e-3},
    ],
    "DDPG":[
        {"hid":128,"nl":2,"la":1e-4,"lc":1e-3,"gamma":0.99,"tau":0.005,"wd":1e-3},
        {"hid":256,"nl":2,"la":5e-5,"lc":5e-4,"gamma":0.99,"tau":0.005,"wd":1e-3},
    ],
}

best_cfgs={}
for name,configs in HP_GRID.items():
    print(f"\n[{name}] Testing {len(configs)} configs ({MAX_EP_SEARCH} ep each)...")
    results=[]
    for i,cfg in enumerate(configs):
        agent,vsr,vcr=train_agent(name,cfg,MAX_EP_SEARCH,f"-cfg{i+1}")
        results.append((vsr,vcr,cfg,i+1))
        del agent; gc.collect(); torch.cuda.empty_cache()
    results.sort(key=lambda x:x[0],reverse=True)
    best_vsr,best_vcr,best_cfg,best_i=results[0]
    best_cfgs[name]=best_cfg
    print(f"  [{name}] Best: config {best_i} val_SR={best_vsr:.3f} val_CR={best_vcr:.1f}%")

print("\nHP search complete. Best configs:")
for n,c in best_cfgs.items(): print(f"  {n}: {c}")

In [ ]:
print("="*60)
print("PHASE 2: Final Training with Best HPs")
print("="*60)

trained={}; val_srs={}; val_crs={}
for name,cfg in best_cfgs.items():
    print(f"\n[{name}] Final training ({MAX_EP_FINAL} ep max)...")
    agent,vsr,vcr=train_agent(name,cfg,MAX_EP_FINAL)
    trained[name]=agent; val_srs[name]=vsr; val_crs[name]=vcr

print("\nVal Summary (2019-2022 bear market val, cost=True):")
for n,v in val_srs.items():
    print(f"  {n:6s}: SR={v:.3f}  CR={val_crs[n]:.1f}%")
print(f"  Best: {max(val_srs,key=val_srs.get)}")

In [ ]:
print("\n"+"="*65)
print(f"FINAL BENCHMARK  {ASSET}  Test:2023+  [cost=True]")
print("="*65)

all_vals={}
bnh=make_env(test_S,test_R,rand=False,cost=False); bnh.reset()
for _ in range(bnh.T-1): bnh.step(2)
bm=bnh.metrics(); bsr=sharpe(bnh.vals); all_vals["B&H"]=bnh.vals
print(f"  {'B&H':8s} | CR={bm['CR%']:7.1f}% SR={bm['SR']:5.3f} MDD={bm['MDD%']:5.1f}% (baseline)")

rows=[{"Model":"B&H",**bm,"ValSR":"--","ValCR":"--"}]
for name,agent in trained.items():
    env=make_env(test_S,test_R,rand=False,cost=True)
    obs,_=env.reset(); done=False
    while not done: obs,_,done,_,_=env.step(agent.greedy(obs))
    tsr=sharpe(env.vals); m=env.metrics(); all_vals[name]=env.vals
    beat="BEAT" if m["CR%"]>bm["CR%"] else "----"
    print(f"  {name:8s} | CR={m['CR%']:7.1f}% SR={m['SR']:5.3f} MDD={m['MDD%']:5.1f}%  [{beat}] ValSR={val_srs[name]:.3f}")
    rows.append({"Model":name,**m,"ValSR":round(val_srs[name],3),"ValCR":round(val_crs[name],1)})

df_res=pd.DataFrame(rows)
df_res.to_csv(f"results_{ASSET}_v3.csv",index=False)
print(f"\nSaved results_{ASSET}_v3.csv")
print("\n"+df_res.to_string(index=False))
PAPER={"B&H":"16.45%","PG":"13.96%","PPO":"2.54%","DQL":"52.62%","DDPG":"54.96%"}
print("\nVS PAPER:")
for _,r in df_res.iterrows():
    m=r["Model"]; print(f"  {m:6s} paper={PAPER.get(m,'--'):8s} yours={r['CR%']:.1f}%")

In [ ]:
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec

CLR={"B&H":"black","PG":"#1f77b4","PPO":"#ff7f0e","DQL":"#2ca02c","DDPG":"#d62728"}
fig=plt.figure(figsize=(20,22))
gs=gridspec.GridSpec(3,2,figure=fig,hspace=0.4,wspace=0.35)

ax1=fig.add_subplot(gs[0,0])
ax1.set_title("Portfolio Value - Test 2023+ (cost=True)",fontsize=13,fontweight="bold")
for name,vals in all_vals.items():
    ls="--" if name=="B&H" else "-"; lw=2 if name=="B&H" else 1.5
    ax1.plot(vals,label=name,color=CLR.get(name,"gray"),ls=ls,lw=lw)
ax1.set_xlabel("Trading Days"); ax1.set_ylabel("Portfolio Value ($)")
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_:f"${x/1e3:.0f}K"))
ax1.legend(fontsize=10); ax1.grid(True,alpha=0.3)

ax2=fig.add_subplot(gs[0,1])
ax2.set_title("Risk vs Return",fontsize=13,fontweight="bold")
for _,row in df_res.iterrows():
    m=row["Model"]; cr=row["CR%"]; mdd=row["MDD%"]
    ax2.scatter(mdd,cr,s=300,color=CLR.get(m,"gray"),zorder=5,edgecolors="white",lw=2)
    ax2.annotate(m,(mdd,cr),textcoords="offset points",xytext=(7,4),fontsize=12,fontweight="bold")
ax2.set_xlabel("Max Drawdown % (lower=better)"); ax2.set_ylabel("Cumulative Return %")
ax2.invert_xaxis(); ax2.grid(True,alpha=0.3)
ax2.axhline(y=bm["CR%"],color="black",ls="--",alpha=0.5,label="B&H")
ax2.legend(fontsize=10)

ax3=fig.add_subplot(gs[1,0])
ax3.set_title("Sharpe Ratio",fontsize=13,fontweight="bold")
mods=df_res["Model"].tolist(); srs=df_res["SR"].tolist()
bars=ax3.bar(mods,srs,color=[CLR.get(m,"gray") for m in mods],edgecolor="white",lw=1.5)
for bar,val in zip(bars,srs):
    ax3.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,f"{val:.3f}",
             ha="center",va="bottom",fontsize=10,fontweight="bold")
ax3.axhline(1.0,color="orange",ls="--",alpha=0.7,label="SR=1 (good)")
ax3.axhline(2.0,color="green",ls="--",alpha=0.7,label="SR=2 (excellent)")
ax3.set_ylabel("Sharpe Ratio"); ax3.legend(fontsize=9); ax3.grid(True,alpha=0.3,axis="y")

ax4=fig.add_subplot(gs[1,1])
ax4.set_title("Val SR vs Test SR (generalization gap)",fontsize=13,fontweight="bold")
ags=[r for r in df_res.itertuples() if r.Model!="B&H"]
x=range(len(ags)); names=[r.Model for r in ags]
vsrs=[float(r.ValSR) for r in ags]; tsrs=[r.SR for r in ags]
ax4.bar([i-0.2 for i in x],vsrs,0.35,label="Val SR (2019-2022)",color="steelblue",alpha=0.8)
ax4.bar([i+0.2 for i in x],tsrs,0.35,label="Test SR (2023+)",color="coral",alpha=0.8)
ax4.set_xticks(list(x)); ax4.set_xticklabels(names,fontsize=11)
ax4.legend(fontsize=10); ax4.grid(True,alpha=0.3,axis="y")
ax4.set_ylabel("Sharpe Ratio")

ax5=fig.add_subplot(gs[2,0])
ax5.set_title("Max Drawdown %",fontsize=13,fontweight="bold")
mdds=df_res["MDD%"].tolist()
bars2=ax5.bar(mods,mdds,color=[CLR.get(m,"gray") for m in mods],edgecolor="white",lw=1.5)
for bar,val in zip(bars2,mdds):
    ax5.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.4,f"{val:.1f}%",
             ha="center",va="bottom",fontsize=10,fontweight="bold")
ax5.axhline(bm["MDD%"],color="black",ls="--",alpha=0.5,label=f"B&H={bm['MDD%']:.1f}%")
ax5.set_ylabel("MDD %"); ax5.legend(fontsize=9); ax5.grid(True,alpha=0.3,axis="y")

ax6=fig.add_subplot(gs[2,1]); ax6.axis("off")
ax6.set_title("Benchmark vs Paper",fontsize=13,fontweight="bold")
PAPER={"B&H":"16.45%","PG":"13.96%","PPO":"2.54%","DQL":"52.62%","DDPG":"54.96%"}
tdata=[[r["Model"],f"{r['CR%']:.1f}%",f"{r['SR']:.3f}",f"{r['MDD%']:.1f}%",PAPER.get(r["Model"],"--")]
       for _,r in df_res.iterrows()]
tbl=ax6.table(cellText=tdata,colLabels=["Model","CR%","SR","MDD%","Paper CR%"],
              cellLoc="center",loc="center",bbox=[0,0,1,0.85])
tbl.auto_set_font_size(False); tbl.set_fontsize(11)
for (r,c),cell in tbl.get_celld().items():
    if r==0: cell.set_facecolor("#2c3e50"); cell.set_text_props(color="white",fontweight="bold")
    elif r%2==0: cell.set_facecolor("#ecf0f1")

fig.suptitle(f"DRL Trading {ASSET}  Train:2010-18  Val:2019-22(bear)  Test:2023+\n"
             f"Qwen2.5-3B SEntFiN F1=0.9047 | HP-tuned | NaN-safe | Anti-overfit",
             fontsize=13,fontweight="bold",y=0.99)
out=f"/kaggle/working/dashboard_{ASSET}_v3.png"
plt.savefig(out,dpi=150,bbox_inches="tight"); plt.close()
print(f"Saved {out}")
print("Download: results_{ASSET}_v3.csv + dashboard_{ASSET}_v3.png")